In [ ]:
import json
import sys
from pathlib import Path

import geopandas as geopd
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.colors import hex_to_rgb
from plotly.subplots import make_subplots
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
# --- Repo-specific settings: adjust these when moving the notebook to another repo ---

# territorial_indexes sits next to this notebook's folder; its package `utils` uses absolute
# imports (`from utils.config import ...`), so territorial_indexes itself goes on sys.path.
TERRITORIAL_INDEXES_DIR = Path.cwd().parent / "territorial_indexes"
if str(TERRITORIAL_INDEXES_DIR) not in sys.path:
    sys.path.insert(0, str(TERRITORIAL_INDEXES_DIR))

# Local module: S3/DigitalHub helpers
REPO_ROOT = Path.cwd() if (Path.cwd() / "utils.py").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from utils import get_s3, get_mapping, get_s3
# Input files on S3 (read with get_s3)
VODAFONE_ATTENDENCES_FILE = "vodafone_attendences_new.csv"
COMUNI_GEOJSON_FILE = "TRENTINO-comuni_Vodafone_2023.geojson"

# Output file (next to this notebook): mapping comune ISTAT -> cluster, read by the backend indicator
# "gini-stagionalita" (published on the platform as an artifact with the same name)
CLUSTER_MAPPING_FILE = "mapping_comuni_cluster_stagionalita.json"

# CARICAMENTO DATI

In [ ]:
vodafone_attendences_new = pd.read_csv(get_s3(VODAFONE_ATTENDENCES_FILE))
geojson_comuni_json_data = geopd.read_file(get_s3(COMUNI_GEOJSON_FILE))

In [ ]:
# Vigo e Pozza di Fassa -> San Giovanni di Fassa
FASSA = {"VIGO DI FASSA": "SAN GIOVANNI DI FASSA", "POZZA DI FASSA": "SAN GIOVANNI DI FASSA"}

location_map = dict(
    zip(
        geojson_comuni_json_data["id"].astype(str),
        geojson_comuni_json_data["name"].str.upper(),
    )
)

# presenze turistiche giornaliere Vodafone per comune
tourists = vodafone_attendences_new[
    (vodafone_attendences_new["locType"] == "TN_MKT_AL_3")
    & (vodafone_attendences_new["userProfile"] == "TOURIST")
]
tourists_attendences = (
    tourists.assign(
        date=pd.to_datetime(tourists["date"]),
        comune=tourists["locId"].astype(str).map(location_map).replace(FASSA),
    )
    .groupby(["date", "comune"], as_index=False)["value"]
    .sum()
)

gdf_map = geojson_comuni_json_data.assign(
    comune_upper=geojson_comuni_json_data["name"].str.upper().replace(FASSA)
)

## STAGIONALITÀ

Come in `serie_unificate_stagionalita.ipynb`:
- **serie vere**: presenze turistiche giornaliere (giorno mancante = nessun turista) con media mobile centrata a 7 giorni;
- **componente stagionale**: decomposizione additiva delle serie vere con periodo annuale (usata per il clustering);
- **stagionalità affidabile** se il profilo settimanale senza trend si ripete tra gli anni (correlazione media ≥ `MIN_YOY_CORR`).

In [ ]:
MIN_YOY_CORR = 0.5  # sotto questa soglia l'andamento annuale non si ripete abbastanza: stagionalità non affidabile

# serie vere: giorni mancanti = nessun turista -> 0, media mobile centrata 7 gg (tolti i NaN ai bordi)
ts_real = (
    tourists_attendences.pivot(index="date", columns="comune", values="value")
    .asfreq("D")
    .fillna(0)
    .rolling(7, center=True)
    .mean()
    .dropna()
)

seasonal, yoy = {}, {}
for comune, s in ts_real.items():
    res = seasonal_decompose(s, period=365, extrapolate_trend="period")
    seasonal[comune] = res.seasonal

    # quanto si ripete l'andamento annuale: correlazione media tra gli anni del profilo settimanale senza trend
    detrended = s - res.trend
    weeks = detrended.index.isocalendar().week.clip(upper=52)
    weekly = detrended.groupby([weeks, detrended.index.year]).mean().unstack().dropna()  # 52 settimane x anni
    yoy[comune] = weekly.corr().values[np.triu_indices(weekly.shape[1], 1)].mean()

ts_seasonal = pd.DataFrame(seasonal).rename_axis(columns="comune")
yoy_corr = pd.Series(yoy).rename_axis("comune")
has_seasonality = (yoy_corr >= MIN_YOY_CORR).rename("has_seasonality")

## CLUSTER

Clustering gerarchico sulla correlazione delle componenti stagionali, solo per i comuni con stagionalità.
I comuni senza stagionalità formano un gruppo a sé (`cluster = 0`).

In [ ]:
CLUSTERS = 2
METHOD = "ward"
NON_SEASONAL_CLUSTER = 0
# colore di ogni cluster (hex), usato nelle serie e nella mappa; l'ordine è quello dei pannelli
CLUSTER_COLORS = {1: "#fa9632", 2: "#00b5ce", NON_SEASONAL_CLUSTER: "#fd46c9"}


def normalize(ts):
    """Normalizzazione min-max (0-1) di ogni comune."""
    return (ts - ts.min()) / (ts.max() - ts.min()).replace(0, np.nan)


corr_matrix = normalize(ts_seasonal).loc[:, has_seasonality].corr()

# distanza di Ward sulla correlazione: sqrt(2 * (1 - corr))
dist_matrix = np.sqrt(2 * (1 - corr_matrix)) if METHOD == "ward" else 1 - corr_matrix
Z = linkage(squareform(dist_matrix.values, checks=False), method=METHOD)

cluster = pd.Series(NON_SEASONAL_CLUSTER, index=has_seasonality.index, name="cluster")
cluster[corr_matrix.columns] = fcluster(Z, t=CLUSTERS, criterion="maxclust")

## GRAFICI

Nella cella sotto scegli quali serie mostrare: le serie vere o la sola componente stagionale.
I cluster non cambiano (sono sempre calcolati sulla componente stagionale); dopo aver cambiato la scelta,
riesegui le celle dei grafici.

In [ ]:
PLOT_SERIES = "vere"  # "vere" = presenze turistiche (media mobile 7 gg) | "stagionalita" = componente stagionale

ts_plot, series_label = {
    "vere": (ts_real, "Presenze turistiche (media mobile 7 gg)"),
    "stagionalita": (ts_seasonal, "Componente stagionale"),
}[PLOT_SERIES]
ts_plot = normalize(ts_plot)

In [ ]:
# Linea centrale: mediana del cluster; spread: bande di percentili annidate (non media ± SD). Le serie
# normalizzate stanno in 0-1 e sono asimmetriche (molti valori vicino a 0 in bassa stagione): media ± SD
# uscirebbe da 0-1, i percentili invece restano in 0-1 e seguono l'asimmetria reale della distribuzione.
BANDS = {(0.10, 0.90): 0.15, (0.25, 0.75): 0.3}  # (percentile basso, alto) -> opacità della banda
SHOW_COMUNI = False  # True: sovrappone anche le serie dei singoli comuni

fig_ts = make_subplots(
    rows=len(CLUSTER_COLORS),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=[
        f"Cluster {c}"
        f" - {(cluster == c).sum()} comuni"
        for c in CLUSTER_COLORS
    ],
)
for row, (c, color) in enumerate(CLUSTER_COLORS.items(), start=1):
    r, g, b = hex_to_rgb(color)
    ts_c = ts_plot.loc[:, cluster.index[cluster == c]]
    for (low, high), alpha in BANDS.items():
        fig_ts.add_trace(go.Scatter(
            x=ts_c.index, y=ts_c.quantile(high, axis=1), mode="lines",
            line=dict(width=0), hoverinfo="skip",
        ), row=row, col=1)
        fig_ts.add_trace(go.Scatter(
            x=ts_c.index, y=ts_c.quantile(low, axis=1), mode="lines",
            line=dict(width=0), fill="tonexty", fillcolor=f"rgba({r}, {g}, {b}, {alpha})", hoverinfo="skip",
        ), row=row, col=1)
    if SHOW_COMUNI:
        for comune, values in ts_c.items():
            fig_ts.add_trace(go.Scatter(
                x=values.index, y=values, mode="lines", name=comune,
                line=dict(color=f"rgba({r}, {g}, {b}, 0.2)", width=1),
                hovertemplate="%{fullData.name}<br>%{x|%d/%m/%Y}: %{y:.2f}<extra></extra>",
            ), row=row, col=1)
    fig_ts.add_trace(go.Scatter(
        x=ts_c.index, y=ts_c.median(axis=1), mode="lines", name=f"Mediana cluster {c}",
        line=dict(color=color, width=2.5),
        hovertemplate="%{fullData.name}<br>%{x|%d/%m/%Y}: %{y:.2f}<extra></extra>",
    ), row=row, col=1)

fig_ts.update_yaxes(range=[0, 1], title_text="Valore normalizzato")
fig_ts.update_layout(
    title=dict(
        text=f"{series_label} - serie normalizzate (min-max) per cluster",
        subtitle=dict(text="Linea: mediana del cluster - bande: " + ", ".join(
            f"{high - low:.0%} centrale dei comuni ({low * 100:.0f}°-{high * 100:.0f}° percentile)" for low, high in BANDS
        )),
    ),
    showlegend=False,
    height=900,
    margin=dict(t=110),
    template="plotly_white",
)
fig_ts.show()

In [ ]:
# Componente comune: per ogni giorno il minimo tra le mediane dei cluster (ogni cluster pesa uguale),
# cioè la parte dell'andamento presente in tutti e tre (es. Capodanno, Ferragosto).
# Togliendola resta la parte specifica di ogni comune: positiva dove supera la parte comune,
# negativa dove ne ha meno. Togliere lo stesso valore a tutti i comuni sposta la mediana di quel valore,
# quindi la mediana di un cluster è esattamente (mediana - parte comune):
# >= 0, e 0 nei giorni in cui è proprio quel cluster a definire la parte comune.
cluster_medians = ts_plot.T.groupby(cluster).median().T
common = cluster_medians.min(axis=1)
specific_medians = cluster_medians.sub(common, axis=0)

# unico fattore per tutti i comuni, preso dalle mediane (non da un singolo comune): le mediane stanno
# in 0-1 (1 = la differenza specifica più forte) e restano confrontabili tra cluster
scale = specific_medians.max().max()
ts_specific = ts_plot.sub(common, axis=0) / scale
specific_medians /= scale

fig_specific = make_subplots(
    rows=len(CLUSTER_COLORS),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=[
        f"Cluster {c}"
        f" - {(cluster == c).sum()} comuni"
        for c in CLUSTER_COLORS
    ],
)
for row, (c, color) in enumerate(CLUSTER_COLORS.items(), start=1):
    r, g, b = hex_to_rgb(color)
    ts_c = ts_specific.loc[:, cluster.index[cluster == c]]
    # banda: 50% centrale dei comuni del cluster (25°-75° percentile)
    fig_specific.add_trace(go.Scatter(
        x=ts_c.index, y=ts_c.quantile(0.75, axis=1), mode="lines",
        line=dict(width=0), hoverinfo="skip",
    ), row=row, col=1)
    fig_specific.add_trace(go.Scatter(
        x=ts_c.index, y=ts_c.quantile(0.25, axis=1), mode="lines",
        line=dict(width=0), fill="tonexty", fillcolor=f"rgba({r}, {g}, {b}, 0.2)", hoverinfo="skip",
    ), row=row, col=1)
    fig_specific.add_trace(go.Scatter(
        x=specific_medians.index, y=specific_medians[c], mode="lines", name=f"Mediana cluster {c}",
        line=dict(color=color, width=2.5),
        hovertemplate="%{fullData.name}<br>%{x|%d/%m/%Y}: %{y:.2f}<extra></extra>",
    ), row=row, col=1)

fig_specific.add_hline(
    y=0, line=dict(color="#6B7280", width=1, dash="dash"),
    annotation_text="parte comune", annotation_position="bottom right",
    annotation_font=dict(color="#6B7280", size=11),
)
fig_specific.update_yaxes(matches="y", title_text="Parte specifica")
fig_specific.update_layout(
    title=dict(
        text=f"{series_label} - parte specifica: mediana del cluster meno la parte comune ai 3 cluster",
        subtitle=dict(text="1 = differenza più forte - banda: 50% centrale dei comuni (25°-75° percentile)"),
    ),
    showlegend=False,
    height=900,
    margin=dict(t=110),
    template="plotly_white",
)
fig_specific.show()

## INDICE DI GINI

Stessa logica dell'indicatore della piattaforma `gini-stagionalita` (`TouristPresenceGiniIndicator`;
`normalized_gini` qui sotto è una copia del suo `_gini`): Gini normalizzato delle presenze giornaliere di un periodo, calcolato solo se
sono disponibili almeno l'80% dei giorni. Qui ogni anno di `YEARS` è un periodo; poi si fa la media sugli anni validi.

In [ ]:
YEARS = [2023, 2024, 2025]
MIN_COVERAGE = 0.8  # come TouristPresenceGiniIndicator._MIN_COVERAGE

# presenze giornaliere grezze: i giorni mancanti restano NaN (come nel panel del backend)
daily = tourists_attendences.pivot(index="date", columns="comune", values="value")


def normalized_gini(values):
    """Copia di TouristPresenceGiniIndicator._gini (trentino_indicators.py): Gini normalizzato di valori non negativi,
    0 = distribuzione uniforme, 1 = massima concentrazione. NaN con meno di 2 valori, valori negativi o somma nulla."""
    values = np.asarray(values, dtype=float)
    if values.size < 2 or not np.isfinite(values).all() or (values < 0).any() or values.sum() <= 0:
        return np.nan

    values = np.sort(values)  # va calcolato sulla serie ordinata
    count = values.size
    weighted_sum = np.dot(2 * np.arange(1, count + 1) - count - 1, values)
    return float(weighted_sum / (count * values.sum()) * count / (count - 1))  # normalizzato: indice tra 0 e 1


def gini_period(s):
    """Gini di un periodo come nel backend; NaN (invece di -1) se i giorni disponibili sono meno di MIN_COVERAGE."""
    return normalized_gini(s.dropna()) if s.notna().mean() >= MIN_COVERAGE else np.nan


gini_years = pd.DataFrame({
    year: daily.reindex(pd.date_range(f"{year}-01-01", f"{year}-12-31", freq="D")).apply(gini_period)
    for year in YEARS
})
gini_comune = gini_years.mean(axis=1).rename("gini")

## CLUSTER E INDICE DI GINI

Nella mappa il colore indica il cluster e la sua intensità l'indice di Gini, su una scala fissa 0-1
(0 = colore schiarito, 0.5 = colore del cluster, 1 = colore scurito): non dipende dal massimo del dataset,
quindi le mappe di dataset diversi restano confrontabili. I comuni senza Gini (dati insufficienti) non sono colorati.

In [ ]:
def gini_colorscale(color, tint=0.85, shade=0.45):
    """Scala fissa sul Gini 0-1: colore schiarito (0) -> colore del cluster (0.5) -> colore scurito (1)."""
    rgb = np.array(hex_to_rgb(color))
    stops = [(0, rgb + (255 - rgb) * tint), (0.5, rgb), (1, rgb * (1 - shade))]
    return [[z, "rgb({:.0f}, {:.0f}, {:.0f})".format(*v)] for z, v in stops]


df_cluster_gini = cluster.to_frame().join(gini_comune).reset_index()
map_cg = gdf_map.merge(df_cluster_gini, left_on="comune_upper", right_on="comune")

fig_cg = go.Figure()
for c, color in CLUSTER_COLORS.items():
    sub = map_cg[map_cg["cluster"] == c]
    fig_cg.add_trace(go.Choropleth(
        geojson=sub.__geo_interface__,
        locations=sub.index,
        z=sub["gini"],
        zmin=0,
        zmax=1,
        colorscale=gini_colorscale(color),
        showscale=False,
        text=sub["name"],
        hovertemplate=f"%{{text}}<br>Cluster {c}<br>Gini: %{{z:.2f}}<extra></extra>",
    ))

fig_cg.update_geos(fitbounds="locations", visible=False)
fig_cg.update_layout(
    title="Cluster turistici (colore) e indice di Gini (intensità)",
    width=1000,
    height=700,
    template="plotly_white",
)
fig_cg.show()

## MAPPING COMUNE → CLUSTER PER IL BACKEND

Il clustering è per area Vodafone, il backend lavora sui comuni ISTAT: ogni comune prende il cluster della sua area.
Il file `CLUSTER_MAPPING_FILE` (`{"022001": 1, ...}`) è letto dall'indicatore `gini-stagionalita` del backend.

In [ ]:
# Vigo e Pozza di Fassa (aree Vodafone) sono oggi il comune di San Giovanni di Fassa: nel mapping hanno "NA"
SAN_GIOVANNI_DI_FASSA = "022250"

# area Vodafone -> codici ISTAT dei suoi comuni
comune_cluster = {
    SAN_GIOVANNI_DI_FASSA if c == "NA" else f"{int(c):06d}": int(cluster[FASSA.get(area, area)])
    for area, comuni in get_mapping("mapping_comuni_into_vodafone_Trento.json").items()
    for c in comuni
}
assert len(comune_cluster) == 166, f"{len(comune_cluster)} comuni invece dei 166 del Trentino"

with open(CLUSTER_MAPPING_FILE, "w", encoding="utf-8") as f:
    json.dump(dict(sorted(comune_cluster.items())), f, indent=1)